# 🧪 ADK Application Testing

This notebook demonstrates how to test an ADK (Agent Development Kit) application.
It covers both local and remote testing, both with Agent Engine and Cloud Run.

> **Note**: This notebook assumes that the agent files are stored in the `app` folder. If your agent files are located in a different directory, please update all relevant file paths and references accordingly.

## Set Up Your Environment

> **Note:** For best results, use the same `.venv` created for local development with `uv` to ensure dependency compatibility and avoid environment-related issues.

In [ ]:
# Uncomment the following lines if you're not using the virtual environment created by uv
# import sys

# sys.path.append("../")
# !pip install google-cloud-aiplatform a2a-sdk --upgrade

### Import libraries

In [37]:
import json

import requests
import vertexai

In [38]:
import os

from dotenv import load_dotenv

# Load environment variables from the project root's .env file
load_dotenv(dotenv_path="../.env")

# Verify that the variables have loaded successfully (without displaying sensitive secrets)
project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
model = os.getenv("MODEL")
REGION = os.getenv("GOOGLE_CLOUD_REGION")
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION")

print(f"Loaded environment variables for project: {project_id}")
print(f"Using Model: {model}")
print(f"Region: {REGION}")
print(f"Location: {LOCATION}")



Loaded environment variables for project: finops-admin-dev
Using Model: gemini-3.5-flash
Region: europe-west1
Location: global


### Initialize Vertex AI Client

In [39]:
# Initialize the Vertex AI client
client = vertexai.Client(
    location=REGION,
)

## If you are using Agent Engine
See more documentation at [Agent Engine Overview](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview)

### Remote Testing

In [35]:
# Set to None to auto-detect from ./deployment_metadata.json, or specify manually
# "projects/PROJECT_ID/locations/us-central1/reasoningEngines/ENGINE_ID"
REMOTE_RUNTIME_ENGINE_ID = None

if REMOTE_RUNTIME_ENGINE_ID is None:
    try:
        with open("../deployment_metadata.json") as f:
            metadata = json.load(f)
            RUNTIME_ENGINE_ID = metadata.get("remote_agent_runtime_id")
    except (FileNotFoundError, json.JSONDecodeError):
        pass

print(f"Using REASONING_ENGINE_ID: {RUNTIME_ENGINE_ID}")
# Get the existing agent engine
remote_agent_engine = client.agent_engines.get(name=RUNTIME_ENGINE_ID)

Using REASONING_ENGINE_ID: projects/867912466457/locations/europe-west1/reasoningEngines/6161935840940392448


In [40]:
async for event in remote_agent_engine.async_stream_query(
    message="hi!", user_id="test"
):
    print(event)

{'invocation_id': 'e-89e201bf-95b8-4f36-9e94-af7bf142eb31', 'author': 'root_agent', 'actions': {'state_delta': {'_turn_tool_call_count': 0}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': 'root_agent@1'}, 'id': '8bb9da6b-d82e-485e-a72b-3bdabd7db713', 'timestamp': 1782327997.0858715}
{'model_version': 'gemini-3.5-flash', 'content': {'parts': [{'text': 'Hello! I am your FinOps AI Assistant, specialized in helping you analyze, track, and optimize your Google Cloud Platform (GCP) costs. \n\nI have direct access to your GCP billing exports in BigQuery, as well as tools to scan for "zombie" resources (like unattached disks or idle IP addresses), retrieve Cloud Asset Inventory (CAI) metadata, and pull up-to-date recommendations.\n\nHere are a few things I can help you with:\n* **Cost Analysis & Historical Trends**: Breaking down spend by service, SKU, project, or date.\n* **Cost Spikes & Root Cause Analysis**: Investigating unexp

### Local Testing

You can import directly the AgentEngineApp class within your environment. 
To run the agent locally, follow these steps:
1. Make sure all required packages are installed in your environment
2. The recommended approach is to use the same virtual environment created by the 'uv' tool
3. You can set up this environment by running 'make install' from your agent's root directory
4. Then select this kernel (.venv folder in your project) in your Jupyter notebook to ensure all dependencies are available

In [27]:
# Import the actual agent_runtime from your project
from app.agent_runtime_app import agent_runtime

# Set up and initialise the local agent
agent_runtime.set_up()

In [28]:
async for event in agent_runtime.async_stream_query(message="hi!", user_id="test"):
    print(event)


{'invocation_id': 'e-e5a2c893-7e40-4d2c-9080-075434015415', 'author': 'root_agent', 'actions': {'state_delta': {'_turn_tool_call_count': 0}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': 'root_agent@1'}, 'id': 'f497267a-6dd4-40e3-b718-d77d71a55d63', 'timestamp': 1782327220.1053581}


/home/dazbo/localdev/smart-gcp-finops/.venv/lib/python3.13/site-packages/google/adk/models/google_llm.py:199: UserWarning: [EXPERIMENTAL] GeminiContextCacheManager: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  cache_manager = GeminiContextCacheManager(self.api_client)


{'model_version': 'gemini-3.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-3c50f8f1-bc25-4fdd-b4b0-50ca34d175c0', 'args': {'sql': 'SELECT\n  DATE(usage_start_time) as usage_date,\n  project.id as project_id,\n  service.description as service_description,\n  SUM(cost) as daily_cost,\n  currency\nFROM `finops-admin-473520.all_billing_data.gcp_billing_export_v1_0156FA_B1FF4E_6D9ED8`\nWHERE usage_start_time >= TIMESTAMP(DATE_SUB(CURRENT_DATE(), INTERVAL 30 DAY))\nGROUP BY 1, 2, 3, 5\nHAVING daily_cost > 0.1\nORDER BY usage_date ASC;'}, 'name': 'execute_cached_bigquery_sql'}, 'thought_signature': 'AY89a1-sOJGCDzJ3Apeg0YunuXKNElgUgyi2lliVsP-1WFUDqXs24dgNpER1meV0vXFReMLRG9GHfLngyI-MrGcRk1Jf7q4-5XMJ2yONAFDN8wSsK3mJbKW9TnwFfvWWkA4meY9xTPRuW_zOdjC4IUxIF7rpj6zQfb5j3aAbhfAC-eICbjceLZjBQsK-n2chpahz4-rmxAWjGlcpUjlpDjkYLUViLXBZojxd9kcfdAdfmTtVfLNSwMqoYI4YADREmu_QzHC_HBFGltY0NSdzLcqwbWqbG9vS6AeVPR5Np5zgHqlgs9IVzCGAOueX7doBsJo8aCM8_5J2AD-TI9L_k3Yn_To4cWeh0qeHmrzZqDVChYDvl7L_jUakaG_orz3F9

/home/dazbo/localdev/smart-gcp-finops/.venv/lib/python3.13/site-packages/google/adk/models/google_llm.py:199: UserWarning: [EXPERIMENTAL] GeminiContextCacheManager: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  cache_manager = GeminiContextCacheManager(self.api_client)


{'model_version': 'gemini-3.5-flash', 'content': {'parts': [{'text': 'Hello! I\'m your **FinOps AI Assistant**, here to help you monitor, manage, and optimize your Google Cloud Platform spend. \n\nTo help you hit the ground running, I have just completed a consolidated scan of your billing export for the **last 30 days (from 25 May 2026 to 24 June 2026)**. Here is an overview of your active cloud environment:\n\n### 📊 Financial Summary\n*   **Total Period Cost**: **£64.47 GBP** \n*   **Active Currency**: **GBP (£)**\n*   **Primary Spend Driver**: **Vertex AI** (specifically in the `finops-admin-dev` project)\n*   **Active Runaway / Spikes**: We detected a notable cost spike on **June 2nd and June 3rd** where daily Vertex AI costs surged to **£10.78** and **£15.66** respectively.\n\n---\n\n### 🔍 Top Cost Drivers by Project & Service\n| Project ID | Service Description | Total Cost | % of Spend |\n| :--- | :--- | :--- | :--- |\n| **finops-admin-dev** | Vertex AI | £27.98 | 43.4% |\n| **a

## If you are using Cloud Run

#### Remote Testing

For more information about authenticating HTTPS requests to Cloud Run services, see:
[Cloud Run Authentication Documentation](https://cloud.google.com/run/docs/triggering/https-request)

Remote testing involves using a deployed service URL instead of localhost.

Authentication is handled using GCP identity tokens instead of local credentials.

In [ ]:
ID_TOKEN = get_ipython().getoutput("gcloud auth print-identity-token -q")[0]

In [ ]:
SERVICE_URL = "YOUR_SERVICE_URL_HERE"  # Replace with your Cloud Run service URL

You'll need to first create a Session

In [ ]:
user_id = "test_user_123"
session_data = {"state": {"preferred_language": "English", "visit_count": 1}}

session_url = f"{SERVICE_URL}/apps/app/users/{user_id}/sessions"
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {ID_TOKEN}"}

session_response = requests.post(session_url, headers=headers, json=session_data)
print(f"Session creation status code: {session_response.status_code}")
print(f"Session creation response: {session_response.json()}")
session_id = session_response.json()["id"]

Then you will be able to send a message

In [ ]:
message_data = {
    "app_name": "app",
    "user_id": user_id,
    "session_id": session_id,
    "new_message": {"role": "user", "parts": [{"text": "Hello! Weather in New york?"}]},
    "streaming": True,
}

message_url = f"{SERVICE_URL}/run_sse"
message_response = requests.post(
    message_url, headers=headers, json=message_data, stream=True
)

print(f"Message send status code: {message_response.status_code}")

# Print streamed response
for line in message_response.iter_lines():
    if line:
        line_str = line.decode("utf-8")
        if line_str.startswith("data: "):
            event_json = line_str[6:]
            event = json.loads(event_json)
            print(f"Received event: {event}")

### Local Testing

> You can run the application locally via the `make local-backend` command.

#### Create a session
 Create a new session with user preferences and state information


In [ ]:
user_id = "test_user_123"
session_data = {"state": {"preferred_language": "English", "visit_count": 1}}

session_url = f"http://127.0.0.1:8000/apps/app/users/{user_id}/sessions"
headers = {"Content-Type": "application/json"}

session_response = requests.post(session_url, headers=headers, json=session_data)
print(f"Session creation status code: {session_response.status_code}")
print(f"Session creation response: {session_response.json()}")
session_id = session_response.json()["id"]

#### Send a message
Send a message to the backend service and receive a streaming response


In [ ]:
message_data = {
    "app_name": "app",
    "user_id": user_id,
    "session_id": session_id,
    "new_message": {"role": "user", "parts": [{"text": "Hello! Weather in New york?"}]},
    "streaming": True,
}

message_url = "http://127.0.0.1:8000/run_sse"
message_response = requests.post(
    message_url, headers=headers, json=message_data, stream=True
)

print(f"Message send status code: {message_response.status_code}")

# Print streamed response
for line in message_response.iter_lines():
    if line:
        line_str = line.decode("utf-8")
        if line_str.startswith("data: "):
            event_json = line_str[6:]
            event = json.loads(event_json)
            print(f"Received event: {event}")